# Step 11: Content-profile features in the final model

This notebook tests whether content-profile similarity features improve the final search-ranking ensemble. The features measure how closely a candidate's name, brand and category terms match products in the user's history, using cosine similarity in sparse TF-IDF space.

The experiment first selects a feature set for LightGBM, then retrains LightGBM and the neural ranker with the content-profile features and blends their predictions.

| Part | Experiment |
|---|---|
| A. Feature selection | Compare three combinations of content-profile and LSA features against the 55-column LightGBM baseline, selecting by validation NDCG@10 |
| B. Model training and ensemble | Train LightGBM with the selected 65 columns and the neural ranker with 37 features, average the neural predictions across three seeds, and select the blend on validation |

Evaluation reports test NDCG@10, HR@10 and Purchase-HR@10, with paired bootstrap comparisons and separate results for cold, mixed and warm slates.

Inputs: `features.parquet`, the LSA embedding cache, and `feature_content_profile/content_profile_features.parquet`. Outputs include the final candidate rankings and model comparison tables.


# Part A: Feature selection with LightGBM


In [1]:
import os, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
import pyarrow as pa, pyarrow.parquet as pq, pyarrow.dataset as ds
import warnings
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'
SFX = '_smoke' if SMOKE else ''
SEED = 42
CPS_PATH = OUT_DIR / 'feature_content_profile' / f'content_profile_features{SFX}.parquet'     # Step 9
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']
LSA = ['u_sem_lsa_7d', 'u_sem_lsa_all', 'slate_pct_sem']                                   # Step 4 / 4b profile columns
CPS_TOP5 = ['cps_max', 'slate_pct_cps_max', 'slate_pct_cps_query', 'cps_top3_new', 'cps_query']   # Step 9's suggested subset
nU = len(pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user']))
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)   # same 2% as Step 9 smoke
assert CPS_PATH.exists(), f'{CPS_PATH} not found: run Step 9 first'
print(f'{in_scope.sum():,} records in scope' + (' | SMOKE TEST' if SMOKE else ''))

Mounted at /content/drive
173,831 records in scope


## 1. Load the feature table and content-profile columns

The Step 9 table uses `user, wid` as its key and follows the order of `candidates.parquet` and `features.parquet`. The join is checked for completeness and one-to-one matching. Slates are shuffled with the Step 6 seed; slates with no interaction are excluded from the benchmark.


In [2]:
BASE = [c for c in pq.read_schema(OUT_DIR / 'features.parquet').names
        if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
flt = ds.field('user').isin(pa.array(np.flatnonzero(in_scope).astype(np.int32))) if SMOKE else None
feat = ds.dataset(OUT_DIR / 'features.parquet').to_table(columns=['user', 'wid', 'label', 'split'] + BASE,
                                                         filter=flt).to_pandas()
cps = pd.read_parquet(CPS_PATH)
CPS = [c for c in cps.columns if c not in ('user', 'wid')]
assert not set(CPS) & set(BASE), 'Step 9 column names collide with features.parquet'
assert set(CPS_TOP5) <= set(CPS), 'Step 9 table lacks the expected columns'
if not set(LSA) <= set(BASE):
    print(f'WARNING: {sorted(set(LSA) - set(BASE))} not in features.parquet (run Steps 4 and 4b first); continuing without them')
    LSA = [c for c in LSA if c in BASE]
n0 = len(feat)
feat = feat.merge(cps, on=['user', 'wid'], how='left', validate='one_to_one')
assert len(feat) == n0 and feat['cps_max'].notna().all(), 'Step 9 table does not cover features.parquet'
del cps; gc.collect()

feat = feat.iloc[np.lexsort((np.random.default_rng(SEED).random(len(feat)), feat['user'].to_numpy()))]
feat = feat[feat.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)
part = {s: feat.index[feat['split'] == s].to_numpy() for s in ['train', 'val', 'test']}
groups = {s: feat.loc[part[s]].groupby('user', sort=False).size().to_numpy() for s in part}
U = {s: feat['user'].to_numpy()[part[s]] for s in part}
Y = {s: feat['label'].to_numpy()[part[s]] for s in part}
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
print(f'baseline features {len(BASE)} | Step 9 columns {len(CPS)} | candidate rows {len(feat):,} | ' +
      ', '.join(f'{s} {len(g):,} slates' for s, g in groups.items()))

baseline features 55 | Step 9 columns 13 | candidate rows 15,348,717 | train 120,081 slates, val 25,825 slates, test 25,822 slates


## 2. Select the LightGBM feature set

Every variant uses the same rows and Step 6 LightGBM configuration: LambdaRank, `label_gain` [0, 1, 3, 7], learning rate 0.03, up to 1,200 trees and 63 leaves, with early stopping on validation NDCG@10. Only the feature list changes.

| Variant | Features |
|---|---|
| baseline | The 55 Step 6 columns |
| + all 13 | Baseline plus every Step 9 column |
| + all 13, − LSA | All 13 added, with `u_sem_lsa_7d`, `u_sem_lsa_all` and `slate_pct_sem` removed; Step 9 found no loss from removing them once the TF-IDF profiles were included |
| + top 5, − LSA | Baseline without the three LSA columns, plus `cps_max`, `slate_pct_cps_max`, `slate_pct_cps_query`, `cps_top3_new` and `cps_query`; these accounted for the split gain attributed to 12 of the 13 columns in Step 9 |


In [3]:
TREES = 60 if SMOKE else 1200
LGB = dict(objective='lambdarank', n_estimators=TREES, learning_rate=0.03, num_leaves=63, min_child_samples=200,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, random_state=SEED, n_jobs=-1, verbose=-1,
           label_gain=[0, 1, 3, 7])

def slate_metrics(u, lab, score, k=10, seed=0):
    lab = lab.astype(np.float64)
    order = np.lexsort((np.random.default_rng(seed).random(len(u)), -np.asarray(score, np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    gid = np.repeat(np.arange(len(start)), size)
    disc = np.where(rank < k, 1.0 / np.log2(rank + 2), 0.0)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    idcg = np.bincount(gid, weights=(2 ** lab[np.lexsort((-lab, u))] - 1) * disc)
    has_buy = np.bincount(gid, weights=lab_s == 3) > 0
    return {'user': u_s[start], 'ndcg': dcg / idcg, 'hit': np.bincount(gid, weights=(lab_s > 0) & (rank < k)) > 0,
            'has_buy': has_buy, 'buy_hit': np.bincount(gid, weights=(lab_s == 3) & (rank < k)) > 0}

def summary(m):
    return {'NDCG@10': m['ndcg'].mean(), 'HR@10': m['hit'].mean(), 'Purchase-HR@10': m['buy_hit'][m['has_buy']].mean()}

BASE_NAME = 'baseline (Step 6 LightGBM)'
NO_LSA = [c for c in BASE if c not in LSA]
VARIANTS = {BASE_NAME: BASE,
            '+ all 13': BASE + CPS,
            '+ all 13, - LSA': NO_LSA + CPS,
            '+ top 5, - LSA': NO_LSA + CPS_TOP5}
grid, SCORES, IMP = {}, {}, {}
for name, cols in VARIANTS.items():
    t0 = time.time()
    X = {s: feat.loc[part[s], cols] for s in part}
    m = lgb.LGBMRanker(**LGB).fit(X['train'], Y['train'], group=groups['train'],
                                  eval_set=[(X['val'], Y['val'])], eval_group=[groups['val']], eval_at=[10],
                                  callbacks=[lgb.early_stopping(50, verbose=False)])
    SCORES[name] = {s: m.predict(X[s]) for s in ('val', 'test')}
    imp = pd.Series(m.booster_.feature_importance('gain'), index=cols); IMP[name] = imp / imp.sum()
    val = summary(slate_metrics(U['val'], Y['val'], SCORES[name]['val']))
    grid[name] = {**{f'val {k}': v for k, v in val.items()}, 'features': len(cols), 'trees': m.best_iteration_,
                  'gain share of Step 9 columns': IMP[name].reindex(CPS).fillna(0).sum(), 'min': (time.time() - t0) / 60}
    print(f"{name}: val NDCG@10 {val['NDCG@10']:.4f} | val Purchase-HR@10 {val['Purchase-HR@10']:.4f} | "
          f"{m.best_iteration_} trees | {grid[name]['min']:.1f} min", flush=True)
    del X, m; gc.collect()
GRID = pd.DataFrame(grid).T
GRID.round(4)

baseline (Step 6 LightGBM): val NDCG@10 0.4153 | val Purchase-HR@10 0.6512 | 893 trees | 11.4 min
+ all 13: val NDCG@10 0.4163 | val Purchase-HR@10 0.6542 | 629 trees | 9.3 min
+ all 13, - LSA: val NDCG@10 0.4172 | val Purchase-HR@10 0.6552 | 670 trees | 10.0 min
+ top 5, - LSA: val NDCG@10 0.4160 | val Purchase-HR@10 0.6532 | 515 trees | 7.6 min


,val NDCG@10,val HR@10,val Purchase-HR@10,features,trees,gain share of Step 9 columns,min
baseline (Step 6 LightGBM),0.4153,0.7167,0.6512,55.0,893.0,0.0000,11.3808
+ all 13,0.4163,0.7191,0.6542,68.0,629.0,0.1376,9.3167
"+ all 13, - LSA",0.4172,0.7213,0.6552,65.0,670.0,0.1344,9.9870
"+ top 5, - LSA",0.4160,0.7193,0.6532,57.0,515.0,0.1277,7.5787


In [4]:
CHOSEN = GRID['val NDCG@10'].idxmax()
print(f'chosen on validation: {CHOSEN}')
print('\nsplit-gain share of each Step 9 column (and the LSA columns it competes with):')
pd.concat({n: IMP[n].reindex(CPS + LSA) for n in VARIANTS}, axis=1).round(4)

chosen on validation: + all 13, - LSA

split-gain share of each Step 9 column (and the LSA columns it competes with):


,baseline (Step 6 LightGBM),+ all 13,"+ all 13, - LSA","+ top 5, - LSA"
cps_max,NaN,0.0679,0.0637,0.0682
cps_mean,NaN,0.0011,0.0012,NaN
cps_query,NaN,0.0052,0.0045,0.0053
cps_long,NaN,0.0011,0.0017,NaN
cps_short,NaN,0.0023,0.0025,NaN
cps_top3_new,NaN,0.0047,0.0048,0.0058
cps_cart,NaN,0.0009,0.0012,NaN
cps_buy,NaN,0.0008,0.0008,NaN
cps_max_cart,NaN,0.0015,0.0015,NaN
cps_max_ord,NaN,0.0007,0.0009,NaN


## 3. Test results with paired bootstrap

Each variant is compared with the baseline on identical test slates using 2,000 bootstrap resamples of whole slates. A positive difference favours the variant; `p` is one-sided. Feature selection uses validation NDCG@10. Test results for the remaining variants are included as supplementary comparisons.


In [5]:
M = {n: slate_metrics(U['test'], Y['test'], SCORES[n]['test']) for n in VARIANTS}

def paired(ma, mb, metric, n_boot=2000, seed=SEED):
    if metric == 'NDCG@10':
        d = ma['ndcg'] - mb['ndcg']
    else:
        d = ma['buy_hit'][ma['has_buy']].astype(float) - mb['buy_hit'][mb['has_buy']].astype(float)
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0, len(d), (200, len(d)))].mean(1) for _ in range(n_boot // 200)])
    return {'difference': d.mean(), 'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
            'p (one-sided)': max((boot <= 0).mean(), 1 / n_boot)}

TEST = pd.DataFrame({n: summary(M[n]) for n in VARIANTS}).T
TESTS = {f'{n} vs baseline [{k}]': paired(M[n], M[BASE_NAME], k)
         for n in VARIANTS if n != BASE_NAME for k in ('NDCG@10', 'Purchase-HR@10')}
print(TEST.round(4).to_string()); print()
pd.DataFrame(TESTS).T.round(5)

                            NDCG@10   HR@10  Purchase-HR@10
baseline (Step 6 LightGBM)   0.4158  0.7192          0.6662
+ all 13                     0.4178  0.7233          0.6724
+ all 13, - LSA              0.4183  0.7227          0.6708
+ top 5, - LSA               0.4165  0.7224          0.6662



,difference,ci_low,ci_high,p (one-sided)
+ all 13 vs baseline [NDCG@10],0.00197,0.00064,0.00326,0.0025
+ all 13 vs baseline [Purchase-HR@10],0.00616,-0.00162,0.01393,0.0575
"+ all 13, - LSA vs baseline [NDCG@10]",0.00243,0.00103,0.00375,0.0005
"+ all 13, - LSA vs baseline [Purchase-HR@10]",0.00454,-0.00325,0.01232,0.1380
"+ top 5, - LSA vs baseline [NDCG@10]",0.00071,-0.00074,0.00206,0.1685
"+ top 5, - LSA vs baseline [Purchase-HR@10]",0.00000,-0.00778,0.00778,0.5180


### Cold, mixed and warm slates

An all-cold slate has no interacted item appearing in any user's history. In an all-warm slate, every interacted item appears in a history; mixed slates contain both. This is the segmentation used in Steps 3 to 7. Step 9's gains were concentrated on all-cold slates, where Step 7 also found most ranking failures.


In [6]:
te = feat.loc[part['test']]
pos = te[te['label'] > 0]
warm = pd.Series(np.isin(pos['wid'].to_numpy(), hist_wids)).groupby(pos['user'].to_numpy()).mean()
warm = warm.reindex(M[BASE_NAME]['user']).to_numpy()
SEGMENT = np.where(warm == 0, 'all cold', np.where(warm == 1, 'all warm', 'mixed'))
rows = []
for seg in ['all cold', 'mixed', 'all warm']:
    mask = SEGMENT == seg
    row = {'segment': seg, 'slates': int(mask.sum())}
    for n in VARIANTS:
        row[n] = M[n]['ndcg'][mask].mean()
    row['chosen - baseline'] = row[CHOSEN] - row[BASE_NAME]
    rows.append(row)
SEG = pd.DataFrame(rows).set_index('segment')
SEG.round(4)

,slates,baseline (Step 6 LightGBM),+ all 13,"+ all 13, - LSA","+ top 5, - LSA",chosen - baseline
segment,,,,,,
all cold,3977,0.1124,0.1232,0.1223,0.1214,0.0100
mixed,3379,0.2702,0.2722,0.2725,0.2719,0.0023
all warm,18466,0.5078,0.5079,0.5087,0.5066,0.0008


# Part B: Retrain both models

## 4. Setup

The Step 6 training pipeline uses the `JD_CPS` switch to join the Step 9 table and set the feature lists. LightGBM uses 65 columns: the original 55, with the three LSA profile columns removed and all 13 `cps_*` columns added. The neural ranker uses its 32 lean features plus the five strongest `cps_*` columns, for 37 features.

A check after setup verifies that the switch took effect. Results are saved to `step6_results_cps.csv` and `step6_test_predictions_cps.parquet`; the original Step 6 files are preserved.


In [16]:
import os
os.environ['JD_CPS'] = '1'   # Step 9 columns on in the pipeline below; must run before the next cell
os.environ.pop('JD_SMOKE', None)


In [2]:
import os, sys, time, gc, copy
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb, warnings
import torch, torch.nn as nn, torch.nn.functional as Fn
if sys.platform == 'darwin':
    torch.set_num_threads(1)     # macOS only: torch and LightGBM ship separate OpenMP runtimes that segfault together
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv, pyarrow.parquet as pq, pyarrow.dataset as ds
import scipy.sparse as sp
from scipy.sparse.linalg import svds
warnings.filterwarnings('ignore', message='.*eval_set.*deprecated')

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
OUT_DIR = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'
SEED, K, L_MAX = 42, 64, 100
if SMOKE:
    MAX_EPOCHS, PATIENCE, BATCH, SEEDS, LGB_TREES = 2, 1, 32, [42, 43], 40
else:
    MAX_EPOCHS, PATIENCE, BATCH, SEEDS, LGB_TREES = 12, 2, 48, [42, 43, 44], 600
USE_CPS = os.environ.get('JD_CPS') == '1'      # Step 11: retrain both models with Step 9's content-profile columns
OUT_SFX = ('_smoke' if SMOKE else '') + ('_cps' if USE_CPS else '')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
USE_AMP = DEVICE == 'cuda'
AMP_DEV = 'cuda' if USE_AMP else 'cpu'
print(f'device {DEVICE}' + (f' ({torch.cuda.get_device_name(0)})' if USE_AMP else '') + (' | SMOKE TEST' if SMOKE else ''))
if DEVICE == 'cpu' and not SMOKE:
    print('WARNING: no GPU. Runtime > Change runtime type > L4 GPU.')

FEATURES_LEAN = [
    'slate_pct_pop', 'i_users_click_7d', 'uc_days_since_any', 'i_users_cart_90d', 'i_users_ord_365d',
    'q_name_tfidf_sim', 'slate_len', 'ubrand_days_since_any', 'c3_click_7d', 'slate_cat3_share',
    'u_days_since_click', 'qc_cat3_prior_max', 'uc_click_7d', 'ucat3_days_since_any', 'q_name_match_frac',
    'slate_pct_text', 'slate_pct_sem', 'c3_ord_per_cart_90d', 'q_meta_tfidf_sim', 'q_sem_lsa_sim',
    'u_days_since_cart', 'q_n_terms', 'uc_cart_90d', 'ucat2_click_7d', 'uc_ord_365d',
    'u_ord_365d', 'u_click_7d', 'u_days_since_ord', 'u_cart_per_click_7d', 'u_ord_per_cart_90d',
    'u_queryless_share_7d', 'u_query_repeat_rate']
SEASONAL = ['cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share']
FEATURES_ALL = [c for c in pq.read_schema(OUT_DIR / 'features.parquet').names
                if c not in ('user', 'wid', 'label', 'split') and c not in SEASONAL]
missing = [f for f in FEATURES_LEAN if f not in FEATURES_ALL]
if missing:
    print(f'WARNING: {missing} not in features.parquet (run Steps 4 and 4b first); continuing without them')
    FEATURES_LEAN = [f for f in FEATURES_LEAN if f in FEATURES_ALL]
if USE_CPS:                                    # Step 9 table, keyed by (user, wid); the LSA profile columns are replaced
    CPS_FILE = OUT_DIR / 'feature_content_profile' / f"content_profile_features{'_smoke' if SMOKE else ''}.parquet"
    CPS_ALL = [c for c in pq.read_schema(CPS_FILE).names if c not in ('user', 'wid')]
    CPS_LEAN = ['cps_max', 'slate_pct_cps_max', 'slate_pct_cps_query', 'cps_top3_new', 'cps_query']
    LSA_PROFILE = ['u_sem_lsa_7d', 'u_sem_lsa_all', 'slate_pct_sem']
    FEATURES_ALL = [c for c in FEATURES_ALL if c not in LSA_PROFILE] + CPS_ALL
    FEATURES_LEAN = FEATURES_LEAN + CPS_LEAN
    print(f'Step 9 columns added: LightGBM {len(CPS_ALL)} (minus {len(LSA_PROFILE)} LSA profile columns), neural {len(CPS_LEAN)}')
print(f'feature sets: neural {len(FEATURES_LEAN)} | LightGBM {len(FEATURES_ALL)}')

Mounted at /content/drive
device cuda (NVIDIA L4)
Step 9 columns added: LightGBM 13 (minus 3 LSA profile columns), neural 5
feature sets: neural 37 | LightGBM 65


In [3]:
assert USE_CPS and not SMOKE, 'JD_CPS not set: run the switch cell above first'
assert len(FEATURES_LEAN) == 37 and len(FEATURES_ALL) == 65, (len(FEATURES_LEAN), len(FEATURES_ALL))
assert OUT_SFX == '_cps', OUT_SFX
print('OK: Step 9 columns on, outputs ->', f'step6_results{OUT_SFX}.csv, step6_test_predictions{OUT_SFX}.parquet')


OK: Step 9 columns on, outputs -> step6_results_cps.csv, step6_test_predictions_cps.parquet


### 4.1 Data, embeddings and history

Candidates are shuffled once within each slate before training and evaluation. This prevents LightGBM's tie handling from exploiting the original list order. The neural architecture is invariant to candidate order.


In [4]:
nU = len(pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user']))
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)
scope_users = np.flatnonzero(in_scope)
dset = ds.dataset(OUT_DIR / 'features.parquet')
flt = ds.field('user').isin(pa.array(scope_users.astype(np.int32))) if SMOKE else None
base_cols = [c for c in dict.fromkeys(FEATURES_ALL + FEATURES_LEAN)   # both models' columns (the neural set keeps slate_pct_sem)
             if c not in (CPS_ALL if USE_CPS else [])]
feat = dset.to_table(columns=['user', 'wid', 'label', 'split'] + base_cols, filter=flt).to_pandas()
if USE_CPS:
    n0 = len(feat)
    feat = feat.merge(pd.read_parquet(CPS_FILE, columns=['user', 'wid'] + CPS_ALL), on=['user', 'wid'],
                      how='left', validate='one_to_one')
    assert len(feat) == n0 and feat['cps_max'].notna().all(), f'{CPS_FILE} does not cover features.parquet'
feat = feat.iloc[np.lexsort((np.random.default_rng(SEED).random(len(feat)), feat['user'].to_numpy()))]
feat = feat[feat.groupby('user')['label'].transform('max') > 0].reset_index(drop=True)
USER = feat['user'].to_numpy().astype(np.int64)
LABEL = feat['label'].to_numpy().astype(np.int64)
SPLIT = feat['split'].to_numpy()
print(f'{len(feat):,} candidate rows | slates: ' +
      ', '.join(f'{s} {feat.loc[SPLIT == s, "user"].nunique():,}' for s in ['train', 'val', 'test']))

15,348,717 candidate rows | slates: train 120,081, val 25,825, test 25,822


In [5]:
sfx = '_smoke' if SMOKE else ''
CACHE = {k: OUT_DIR / f'lsa{K}_{k}{sfx}.npy' for k in ('wids', 'item', 'query')}

def safe_save(path, arr):
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        np.save(f, arr)
    np.load(tmp, mmap_mode='r')                       # verify it reads back before replacing
    os.replace(tmp, path)

def l2rows(M):
    n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1
    return M / n

def build_lsa():
    cw = pd.read_parquet(OUT_DIR / 'candidates.parquet', columns=['user', 'wid'])
    ew = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'wid', 'type'])
    ew = ew[ew['type'] < 3]
    need = np.union1d(cw['wid'].to_numpy()[in_scope[cw['user'].to_numpy()]],
                      ew['wid'].to_numpy()[in_scope[ew['user'].to_numpy()]])
    del cw, ew; gc.collect()

    wid_parts, doc_parts, tok_parts, off = [], [], [], 0
    reader = pacsv.open_csv(DATA_DIR / 'product_meta_data.txt',
        parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
        convert_options=pacsv.ConvertOptions(include_columns=['wid', 'name'],
                                             column_types={'wid': pa.int64(), 'name': pa.string()}))
    for batch in reader:
        tb = pa.Table.from_batches([batch])
        wid = tb['wid'].to_numpy()
        j = np.searchsorted(need, wid)
        keep = (j < len(need)) & (need[np.minimum(j, len(need) - 1)] == wid)
        if not keep.any():
            continue
        tb = tb.filter(pa.array(keep))
        parts = pc.split_pattern(pc.fill_null(tb['name'], '-1'), '\x18')
        flat = pc.list_flatten(parts)
        flat = pc.if_else(pc.equal(flat, ''), '-1', flat)
        ids = np.arange(off, off + len(tb), dtype=np.int64); off += len(tb)
        wid_parts.append(tb['wid'].to_numpy())
        doc_parts.append(np.repeat(ids, pc.list_value_length(parts).to_numpy()))
        tok_parts.append(pc.cast(flat, pa.int64()).to_numpy())
    wids = np.concatenate(wid_parts)
    d_, t_ = np.concatenate(doc_parts), np.concatenate(tok_parts)
    m = t_ != -1; d_, t_ = d_[m], t_[m]
    nD = len(wids)

    vocab = np.unique(t_)
    key = np.unique(d_ * len(vocab) + np.searchsorted(vocab, t_))
    d_, c_ = key // len(vocab), key % len(vocab)
    idf = (np.log((nD + 1) / (np.bincount(c_, minlength=len(vocab)) + 1)) + 1).astype(np.float32)
    norm = np.sqrt(np.bincount(d_, weights=idf[c_] ** 2, minlength=nD)); norm[norm == 0] = 1
    A = sp.csr_matrix((idf[c_] / norm[d_], (d_, c_)), shape=(nD, len(vocab)), dtype=np.float32)
    del d_, c_, key, t_; gc.collect()
    v0 = np.random.default_rng(SEED).random(min(A.shape)).astype(np.float32)
    _, S, Vt = svds(A, k=K, v0=v0, return_singular_vectors='vh')
    Vt = Vt[np.argsort(-S)].astype(np.float32)
    item = np.empty((nD, K), np.float32)
    for s in range(0, nD, 1_000_000):
        item[s:s + 1_000_000] = A[s:s + 1_000_000] @ Vt.T
    item = l2rows(item).astype(np.float16)
    del A; gc.collect()

    users = pd.read_parquet(OUT_DIR / 'users.parquet', columns=['user', 'qry'])
    queries = pd.read_parquet(OUT_DIR / 'queries.parquet')
    qs = users['qry'].map(queries.set_index('qry')['text']).str.split('\x18')
    q_doc = np.repeat(np.arange(len(users), dtype=np.int64), qs.str.len().to_numpy())
    q_tok = pd.to_numeric(qs.explode(), errors='coerce').fillna(-1).to_numpy(np.int64)
    pos = np.searchsorted(vocab, q_tok)
    ok = (q_tok != -1) & (pos < len(vocab)) & (vocab[np.minimum(pos, len(vocab) - 1)] == q_tok)
    kq = np.unique(q_doc[ok] * len(vocab) + pos[ok])
    Q = sp.csr_matrix((idf[kq % len(vocab)], (kq // len(vocab), kq % len(vocab))),
                      shape=(len(users), len(vocab)), dtype=np.float32)
    qemb = l2rows(Q @ Vt.T).astype(np.float16)
    return wids, item, qemb

t0 = time.time()
if all(p.exists() for p in CACHE.values()):
    emb_wids, item_emb, q_emb = (np.load(CACHE[k]) for k in ('wids', 'item', 'query'))
    print(f'loaded cached embeddings in {time.time() - t0:.0f}s')
else:
    emb_wids, item_emb, q_emb = build_lsa()
    for k, a in zip(('wids', 'item', 'query'), (emb_wids, item_emb, q_emb)):
        safe_save(CACHE[k], a)
    print(f'built and cached embeddings in {time.time() - t0:.0f}s')
print(f'item embeddings {item_emb.shape} | query embeddings {q_emb.shape}')
emb_index = pd.Index(emb_wids)
DOC = emb_index.get_indexer(feat['wid']) + 1            # 0 = no name / unknown (zero vector)

loaded cached embeddings in 47s
item embeddings (11587364, 64) | query embeddings (173831, 64)


In [6]:
ev = pd.read_parquet(OUT_DIR / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days'])
ev = ev[(ev['type'] < 3) & in_scope[ev['user'].to_numpy()]]
ev = ev.sort_values(['user', 'pos'], kind='stable')
slot = ev.groupby('user').cumcount(ascending=False).to_numpy()      # 0 = most recent
ev, slot = ev[slot < L_MAX], slot[slot < L_MAX]
u_ev = ev['user'].to_numpy()
H_DOC = np.zeros((nU, L_MAX), np.int64); H_TYPE = np.zeros((nU, L_MAX), np.int64)
H_AGE = np.zeros((nU, L_MAX), np.float32); H_MASK = np.zeros((nU, L_MAX), bool)
H_DOC[u_ev, slot] = emb_index.get_indexer(ev['wid']) + 1
H_TYPE[u_ev, slot] = ev['type'].to_numpy()
H_AGE[u_ev, slot] = np.log1p(ev['age_days'].to_numpy())
H_MASK[u_ev, slot] = True
del ev, slot, u_ev; gc.collect()

used = np.unique(np.r_[0, DOC, H_DOC[H_MASK]])
E_full = np.vstack([np.zeros((1, K), np.float16), item_emb])
E = torch.from_numpy(E_full[used]).to(DEVICE)
DOC = np.searchsorted(used, DOC); H_DOC = np.searchsorted(used, H_DOC)
QE = torch.from_numpy(q_emb).to(DEVICE)
del E_full, item_emb; gc.collect()

def build_X(cols):
    X = feat[cols].to_numpy(np.float32, copy=True)
    nan = np.isnan(X)
    np.copysign(np.log1p(np.abs(X)), X, out=X)
    tr = SPLIT == 'train'
    mu = np.nanmean(X[tr], 0); sd = np.nanstd(X[tr], 0); sd[~(sd > 1e-6)] = 1; mu = np.nan_to_num(mu)
    X -= mu; X /= sd; X[nan] = 0
    return torch.from_numpy(np.concatenate([X, nan[:, nan.any(0)].astype(np.float32)], 1))

def make_split(name):
    idx = np.flatnonzero(SPLIT == name)
    u = USER[idx]
    start = np.r_[0, np.flatnonzero(u[1:] != u[:-1]) + 1]
    return {'idx': idx, 'start': start, 'size': np.diff(np.r_[start, len(idx)]), 'user': u[start]}

SPLITS = {s: make_split(s) for s in ['train', 'val', 'test']}
X_NN = build_X(FEATURES_LEAN)
LABEL_T, DOC_T = torch.from_numpy(LABEL), torch.from_numpy(DOC)
H_DOC_T, H_TYPE_T = torch.from_numpy(H_DOC), torch.from_numpy(H_TYPE)
H_AGE_T, H_MASK_T = torch.from_numpy(H_AGE), torch.from_numpy(H_MASK)
print(f'history up to {L_MAX} events | embedding table {E.shape[0]:,} x {K} on {DEVICE} | neural input width {X_NN.shape[1]}')

history up to 100 events | embedding table 9,782,119 x 64 on cuda | neural input width 49


### 4.2 Per-slate metrics

`slate_metrics` returns one value per slate, sorted by user. All models use the same slates for evaluation and paired bootstrap comparisons. The ranking metrics follow the earlier steps. Purchase rank records the position of the first purchased item, with 1 indicating the top position.


In [7]:
def slate_metrics(u, lab, score, k=10, seed=0):
    lab = lab.astype(np.float64)
    r = np.random.default_rng(seed).random(len(u))
    order = np.lexsort((r, -np.asarray(score, dtype=np.float64), u))
    u_s, lab_s = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(u_s[1:] != u_s[:-1]) + 1]
    size = np.diff(np.r_[start, len(u_s)])
    rank = np.arange(len(u_s)) - np.repeat(start, size)
    top = rank < k
    disc = np.where(top, 1.0 / np.log2(rank + 2), 0.0)
    gid = np.repeat(np.arange(len(start)), size)
    dcg = np.bincount(gid, weights=(2 ** lab_s - 1) * disc)
    idcg = np.bincount(gid, weights=(2 ** lab[np.lexsort((-lab, u))] - 1) * disc)
    has_buy = np.bincount(gid, weights=(lab_s == 3)) > 0
    buy_rank = np.minimum.reduceat(np.where(lab_s == 3, rank + 1, 10 ** 6), start).astype(float)
    buy_rank[~has_buy] = np.nan
    return {'ndcg': dcg / idcg, 'hit': np.bincount(gid, weights=((lab_s > 0) & top)) > 0,
            'has_buy': has_buy, 'buy_hit': np.bincount(gid, weights=((lab_s == 3) & top)) > 0, 'buy_rank': buy_rank}

def evaluate(u, lab, score):
    m = slate_metrics(u, lab, score)
    return {'NDCG@10': m['ndcg'].mean(), 'HR@10': m['hit'].mean(), 'Purchase-HR@10': m['buy_hit'][m['has_buy']].mean(),
            'median purchase rank': np.nanmedian(m['buy_rank']), 'slates': len(m['ndcg'])}

def split_eval(name, score):
    S = SPLITS[name]
    return evaluate(USER[S['idx']], LABEL[S['idx']], score)
print('metrics defined')

metrics defined


## 5. Neural ranker training and tuning

The architecture follows Step 5 model C: 32 features plus the five strongest `cps_*` columns, embeddings, history attention, slate context and an ordinal head. The grid varies history length L, width d and the listwise loss weight. Training runs for up to 12 epochs with cosine learning-rate decay and stops after two epochs without an improvement in validation NDCG@10.


In [8]:
class Ranker(nn.Module):
    def __init__(self, n_feat, d=128, k=K, h=64):
        super().__init__()
        self.feat = nn.Sequential(nn.Linear(n_feat, d), nn.ReLU(), nn.Linear(d, d), nn.ReLU())
        self.item, self.qry = nn.Linear(k, h), nn.Linear(k, h)
        self.type_emb, self.age = nn.Embedding(3, 8), nn.Linear(1, 8)
        self.hist = nn.Linear(h + 16, h)
        self.att_h, self.att_c, self.att_p = nn.Linear(h, h), nn.Linear(h, h, bias=False), nn.Linear(h, h, bias=False)
        self.att_out = nn.Linear(h, 1)
        self.enc = nn.Sequential(nn.Linear(d + 3 * h, d), nn.ReLU())
        self.head = nn.Sequential(nn.Linear(3 * d, d), nn.ReLU(), nn.Dropout(0.1), nn.Linear(d, 1))
        self.theta0, self.dtheta = nn.Parameter(torch.tensor(4.0)), nn.Parameter(torch.tensor([0.87, 1.28]))

    def thresholds(self):
        return torch.cat([self.theta0.view(1), self.theta0 + torch.cumsum(Fn.softplus(self.dtheta), 0)])

    def forward(self, b):
        c = self.item(E[b['doc']].float())
        q = self.qry(QE[b['quser']].float())[:, None].expand(-1, c.shape[1], -1)
        hv = self.hist(torch.cat([self.item(E[b['hdoc']].float()), self.type_emb(b['htype']),
                                  self.age(b['hage'][..., None])], -1))
        pre = self.att_h(hv)[:, None] + self.att_c(c)[:, :, None] + self.att_p(hv[:, None] * c[:, :, None])
        a = self.att_out(torch.relu(pre)).squeeze(-1).float().masked_fill(~b['hmask'][:, None, :], -1e4)
        w = torch.softmax(a, -1) * b['hmask'].any(-1)[:, None, None]
        interest = torch.einsum('bml,bld->bmd', w.to(hv.dtype), hv)
        z = self.enc(torch.cat([self.feat(b['X']), c, q, interest], -1))
        m = b['mask'][..., None].to(z.dtype)
        ctx = ((z * m).sum(1, keepdim=True) / m.sum(1, keepdim=True).clamp(min=1)).expand_as(z)
        return self.head(torch.cat([z, ctx, z - ctx], -1)).squeeze(-1)

def make_batch(S, slates, L):
    st, sz = S['start'][slates], S['size'][slates]
    B, M = len(slates), int(sz.max())
    local = np.concatenate([np.arange(a, a + n) for a, n in zip(st, sz)])
    rows = S['idx'][local]; rows_t = torch.from_numpy(rows)
    bi = torch.from_numpy(np.repeat(np.arange(B), sz)); pi = torch.from_numpy(local - np.repeat(st, sz))
    def pad(src):
        out = torch.zeros((B, M) + tuple(src.shape[1:]), dtype=src.dtype)
        out[bi, pi] = src[rows_t]
        return out
    users = S['user'][slates]
    mask = torch.zeros(B, M, dtype=torch.bool); mask[bi, pi] = True
    b = {'X': pad(X_NN), 'doc': pad(DOC_T), 'y': pad(LABEL_T), 'mask': mask, 'quser': torch.from_numpy(users),
         'hdoc': H_DOC_T[users, :L], 'htype': H_TYPE_T[users, :L], 'hage': H_AGE_T[users, :L], 'hmask': H_MASK_T[users, :L]}
    b = {k: v.to(DEVICE, non_blocking=True) for k, v in b.items()}
    b['rows'] = rows
    return b

def loss_fn(s, b, theta, lw):
    m, y = b['mask'], b['y']
    tgt = (y[..., None] >= torch.arange(1, 4, device=s.device)).float()
    bce = Fn.binary_cross_entropy_with_logits(s[..., None] - theta, tgt, reduction='none').sum(-1)
    bce = (bce * m).sum() / m.sum()
    gain = (2.0 ** y - 1) * m
    p = gain / gain.sum(1, keepdim=True).clamp(min=1e-9)
    return bce + lw * -(p * torch.log_softmax(s.masked_fill(~m, -1e4), 1)).sum(1).mean()

@torch.no_grad()
def ordinal_probs(model, name, L, bs=128):
    """P(Y>=1), P(Y>=2), P(Y>=3) for every row of a split, in split order."""
    model.eval(); S = SPLITS[name]
    s_out = np.empty(len(S['idx']), np.float32)
    pos_of = np.empty(len(USER), np.int64); pos_of[S['idx']] = np.arange(len(S['idx']))
    for i in range(0, len(S['start']), bs):
        b = make_batch(S, np.arange(i, min(i + bs, len(S['start']))), L)
        with torch.autocast(device_type=AMP_DEV, dtype=torch.float16, enabled=USE_AMP):
            s = model(b)
        s_out[pos_of[b['rows']]] = s.float()[b['mask']].cpu().numpy()
    theta = model.thresholds().detach().float().cpu().numpy()
    return 1 / (1 + np.exp(-(s_out[:, None] - theta)))

def train_nn(cfg, seed):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    model = Ranker(X_NN.shape[1], d=cfg['d']).to(DEVICE)
    TR = SPLITS['train']; steps = int(np.ceil(len(TR['start']) / BATCH))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=1e-5)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS * steps)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    best, best_state, bad, log = -1.0, None, 0, []
    for ep in range(MAX_EPOCHS):
        model.train(); t0 = time.time()
        perm = rng.permutation(len(TR['start']))
        for i in range(0, len(perm), BATCH):
            b = make_batch(TR, perm[i:i + BATCH], cfg['L'])
            with torch.autocast(device_type=AMP_DEV, dtype=torch.float16, enabled=USE_AMP):
                s = model(b)
            loss = loss_fn(s.float(), b, model.thresholds(), cfg['lw'])
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        val = split_eval('val', ordinal_probs(model, 'val', cfg['L']).sum(1))['NDCG@10']
        log.append(val)
        print(f'    epoch {ep + 1}: val NDCG@10 {val:.4f} ({(time.time() - t0) / 60:.1f} min)')
        if val > best + 1e-4:
            best, best_state, bad = val, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best, len(log)

NN_BASE = {'L': 50, 'd': 128, 'lw': 1.0, 'lr': 1e-3}
GRID = {   # comment out entries to save GPU time
    'base (L=50, d=128, listwise 1.0)': {},
    'longer history (L=100)':           {'L': 100},
    'wider (d=256)':                    {'d': 256},
    'listwise weight 2.0':              {'lw': 2.0},
}
if SMOKE:
    GRID = dict(list(GRID.items())[:2])
nn_grid, nn_probs = {}, {}
for name, delta in GRID.items():
    cfg = {**NN_BASE, **delta}
    print(f'== {name}')
    t0 = time.time()
    model, best, n_ep = train_nn(cfg, SEEDS[0])
    nn_grid[name] = {'val NDCG@10': best, 'epochs': n_ep, 'min': (time.time() - t0) / 60, **cfg}
    nn_probs[(name, SEEDS[0])] = {s: ordinal_probs(model, s, cfg['L']) for s in ('val', 'test')}
    del model; gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()
pd.DataFrame(nn_grid).T.round(4)

== base (L=50, d=128, listwise 1.0)
    epoch 1: val NDCG@10 0.4131 (0.6 min)
    epoch 2: val NDCG@10 0.4162 (0.6 min)
    epoch 3: val NDCG@10 0.4168 (0.6 min)
    epoch 4: val NDCG@10 0.4169 (0.6 min)
    epoch 5: val NDCG@10 0.4182 (0.6 min)
    epoch 6: val NDCG@10 0.4192 (0.6 min)
    epoch 7: val NDCG@10 0.4189 (0.6 min)
    epoch 8: val NDCG@10 0.4190 (0.6 min)
== longer history (L=100)
    epoch 1: val NDCG@10 0.4125 (0.9 min)
    epoch 2: val NDCG@10 0.4158 (0.9 min)
    epoch 3: val NDCG@10 0.4164 (0.9 min)
    epoch 4: val NDCG@10 0.4175 (0.9 min)
    epoch 5: val NDCG@10 0.4179 (0.9 min)
    epoch 6: val NDCG@10 0.4186 (0.9 min)
    epoch 7: val NDCG@10 0.4185 (0.9 min)
    epoch 8: val NDCG@10 0.4189 (0.9 min)
    epoch 9: val NDCG@10 0.4171 (0.9 min)
    epoch 10: val NDCG@10 0.4175 (0.9 min)
== wider (d=256)
    epoch 1: val NDCG@10 0.4134 (0.6 min)
    epoch 2: val NDCG@10 0.4163 (0.6 min)
    epoch 3: val NDCG@10 0.4171 (0.6 min)
    epoch 4: val NDCG@10 0.4178 (0.6 m

,val NDCG@10,epochs,min,L,d,lw,lr
"base (L=50, d=128, listwise 1.0)",0.4192,8.0,4.9997,50.0,128.0,1.0,0.001
longer history (L=100),0.4189,10.0,8.6997,100.0,128.0,1.0,0.001
wider (d=256),0.4192,7.0,4.3979,50.0,256.0,1.0,0.001
listwise weight 2.0,0.4201,10.0,6.2044,50.0,128.0,2.0,0.001


In [9]:
best_nn = max(nn_grid, key=lambda n: nn_grid[n]['val NDCG@10'])
best_cfg = {k: nn_grid[best_nn][k] for k in NN_BASE}
print(f'best setting on validation: {best_nn} {best_cfg}')
for seed in SEEDS[1:]:
    print(f'== {best_nn}, seed {seed}')
    model, _, _ = train_nn(best_cfg, seed)
    nn_probs[(best_nn, seed)] = {s: ordinal_probs(model, s, best_cfg['L']) for s in ('val', 'test')}
    del model; gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()

seed_rows = {seed: split_eval('test', nn_probs[(best_nn, seed)]['test'].sum(1)) for seed in SEEDS}
seed_df = pd.DataFrame(seed_rows).T
print('single-seed test results (score = expected interaction level E[Y] = sum of the three probabilities):')
print(seed_df[['NDCG@10', 'HR@10', 'Purchase-HR@10']].round(4).to_string())
print('mean ± sd: ' + ', '.join(f'{c} {seed_df[c].mean():.4f} ± {seed_df[c].std(ddof=1):.4f}'
                                for c in ['NDCG@10', 'HR@10', 'Purchase-HR@10']))

# the final neural model averages the seeds' probabilities
NN_P = {s: np.mean([nn_probs[(best_nn, seed)][s] for seed in SEEDS], 0) for s in ('val', 'test')}
NN_SCORE = {s: NN_P[s].sum(1) for s in NN_P}
print(f"seed-averaged neural model: val NDCG@10 {split_eval('val', NN_SCORE['val'])['NDCG@10']:.4f}")

best setting on validation: listwise weight 2.0 {'L': 50, 'd': 128, 'lw': 2.0, 'lr': 0.001}
== listwise weight 2.0, seed 43
    epoch 1: val NDCG@10 0.4124 (0.6 min)
    epoch 2: val NDCG@10 0.4151 (0.6 min)
    epoch 3: val NDCG@10 0.4171 (0.6 min)
    epoch 4: val NDCG@10 0.4168 (0.6 min)
    epoch 5: val NDCG@10 0.4189 (0.6 min)
    epoch 6: val NDCG@10 0.4188 (0.6 min)
    epoch 7: val NDCG@10 0.4197 (0.6 min)
    epoch 8: val NDCG@10 0.4193 (0.6 min)
    epoch 9: val NDCG@10 0.4190 (0.6 min)
== listwise weight 2.0, seed 44
    epoch 1: val NDCG@10 0.4133 (0.6 min)
    epoch 2: val NDCG@10 0.4150 (0.6 min)
    epoch 3: val NDCG@10 0.4171 (0.6 min)
    epoch 4: val NDCG@10 0.4184 (0.6 min)
    epoch 5: val NDCG@10 0.4183 (0.6 min)
    epoch 6: val NDCG@10 0.4173 (0.6 min)
single-seed test results (score = expected interaction level E[Y] = sum of the three probabilities):
    NDCG@10   HR@10  Purchase-HR@10
42   0.4198  0.7256          0.6747
43   0.4207  0.7266          0.6743
44   

## 6. LightGBM tuning and purchase weighting

LambdaRank's `label_gain` controls the relevance gains in its NDCG objective. The default `[0, 1, 3, 7]` matches 2^label − 1. Purchase gains of 15 and 31 place more weight on purchases, whose top-10 retrieval is measured by Purchase-HR@10.

The selected gain is tested with two further hyperparameter settings. All variants use the selected 65 columns; the default configuration is retrained as the benchmark.


In [10]:
def lgb_data(name):
    S = SPLITS[name]
    return feat.iloc[S['idx']][FEATURES_ALL], LABEL[S['idx']], S['size']

X_tr, y_tr, g_tr = lgb_data('train'); X_va, y_va, g_va = lgb_data('val'); X_te, _, _ = lgb_data('test')
LGB_BASE = dict(objective='lambdarank', n_estimators=LGB_TREES, learning_rate=0.05, num_leaves=63,
                min_child_samples=200, subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
                random_state=SEED, n_jobs=-1, verbose=-1, label_gain=[0, 1, 3, 7])

def fit_lgb(params):
    m = lgb.LGBMRanker(**params)
    m.fit(X_tr, y_tr, group=g_tr, eval_set=[(X_va, y_va)], eval_group=[g_va], eval_at=[10],
          callbacks=[lgb.early_stopping(50, verbose=False)])
    return m

lgb_grid, lgb_scores = {}, {}
def run_lgb(name, delta):
    params = {**LGB_BASE, **delta}
    t0 = time.time(); m = fit_lgb(params)
    sv, st = m.predict(X_va), m.predict(X_te)
    lgb_scores[name] = {'val': sv, 'test': st}
    lgb_grid[name] = {**{f'val {k}': v for k, v in split_eval('val', sv).items() if k in ('NDCG@10', 'Purchase-HR@10')},
                      'trees': m.best_iteration_, 'min': (time.time() - t0) / 60}
    print(f"  {name}: val NDCG@10 {lgb_grid[name]['val NDCG@10']:.4f} | val Purchase-HR@10 "
          f"{lgb_grid[name]['val Purchase-HR@10']:.4f} | {lgb_grid[name]['min']:.1f} min")

for g in [7, 15, 31]:
    run_lgb(f'label_gain purchase={g}', {'label_gain': [0, 1, 3, g]})
best_gain = max((n for n in lgb_grid if n.startswith('label_gain')), key=lambda n: lgb_grid[n]['val NDCG@10'])
gain_delta = {'label_gain': [0, 1, 3, int(best_gain.split('=')[1])]}
run_lgb(f'{best_gain}, num_leaves 127, min_child 100', {**gain_delta, 'num_leaves': 127, 'min_child_samples': 100})
run_lgb(f'{best_gain}, lr 0.03, more trees', {**gain_delta, 'learning_rate': 0.03, 'n_estimators': 2 * LGB_TREES})

LGB_BENCH = 'label_gain purchase=7'
best_lgb = max(lgb_grid, key=lambda n: lgb_grid[n]['val NDCG@10'])
print(f'\nbest LightGBM on validation: {best_lgb}')
del X_tr, y_tr; gc.collect()
pd.DataFrame(lgb_grid).T.round(4)

  label_gain purchase=7: val NDCG@10 0.4174 | val Purchase-HR@10 0.6529 | 5.6 min
  label_gain purchase=15: val NDCG@10 0.4178 | val Purchase-HR@10 0.6571 | 7.5 min
  label_gain purchase=31: val NDCG@10 0.4177 | val Purchase-HR@10 0.6575 | 7.6 min
  label_gain purchase=15, num_leaves 127, min_child 100: val NDCG@10 0.4175 | val Purchase-HR@10 0.6542 | 6.7 min
  label_gain purchase=15, lr 0.03, more trees: val NDCG@10 0.4169 | val Purchase-HR@10 0.6558 | 9.3 min

best LightGBM on validation: label_gain purchase=15


,val NDCG@10,val Purchase-HR@10,trees,min
label_gain purchase=7,0.4174,0.6529,378.0,5.6184
label_gain purchase=15,0.4178,0.6571,583.0,7.5244
label_gain purchase=31,0.4177,0.6575,600.0,7.5812
"label_gain purchase=15, num_leaves 127, min_child 100",0.4175,0.6542,406.0,6.7496
"label_gain purchase=15, lr 0.03, more trees",0.4169,0.6558,655.0,9.3403


## 7. Ensemble

Before blending, each model's scores are normalized within each slate using either z-scores or percentile ranks. The combined score is w · neural + (1 − w) · LightGBM. The normalization method and weight w are selected on validation.


In [11]:
def within_slate(name, x, how):
    S = SPLITS[name]; st, sz = S['start'], S['size']
    if how == 'rank':
        u = np.repeat(np.arange(len(st)), sz)
        return pd.Series(x).groupby(u).rank(pct=True).to_numpy()
    mu = np.add.reduceat(x, st) / sz
    sd = np.sqrt(np.maximum(np.add.reduceat(x * x, st) / sz - mu ** 2, 0)) + 1e-9
    return (x - np.repeat(mu, sz)) / np.repeat(sd, sz)

LGB_SCORE = {s: lgb_scores[best_lgb][s] for s in ('val', 'test')}
W = np.round(np.linspace(0, 1, 11), 1)
ens_val = {}
for how in ['z-score', 'rank']:
    a, b = within_slate('val', NN_SCORE['val'], how), within_slate('val', LGB_SCORE['val'], how)
    for w in W:
        ens_val[(how, w)] = split_eval('val', w * a + (1 - w) * b)['NDCG@10']
best_how, best_w = max(ens_val, key=ens_val.get)
ENS_SCORE = {s: best_w * within_slate(s, NN_SCORE[s], best_how) + (1 - best_w) * within_slate(s, LGB_SCORE[s], best_how)
             for s in ('val', 'test')}
curve = pd.DataFrame({how: [ens_val[(how, w)] for w in W] for how in ['z-score', 'rank']}, index=W)
curve.index.name = 'weight on neural model'
print(f'chosen on validation: {best_how}, w = {best_w} (val NDCG@10 {ens_val[(best_how, best_w)]:.4f})')
curve.round(4)

chosen on validation: z-score, w = 0.6 (val NDCG@10 0.4226)


,z-score,rank
weight on neural model,,
0.0,0.4178,0.4178
0.1,0.4201,0.4182
0.2,0.4215,0.4194
0.3,0.4219,0.4200
0.4,0.4222,0.4212
0.5,0.4225,0.4221
0.6,0.4226,0.4223
0.7,0.4226,0.4223
0.8,0.4225,0.4222


## 8. Final test results and significance

The retrained models are evaluated on test after selecting their configurations and blend settings on validation. For each paired comparison, 2,000 bootstrap samples resample whole slates with replacement. The mean difference in per-slate NDCG@10 or purchase hit is calculated using the same resampled slates for both models. The distribution gives the 95% interval and one-sided p-value.


In [12]:
TE = SPLITS['test']; te_u, te_lab = USER[TE['idx']], LABEL[TE['idx']]
pop = feat['i_users_ord_365d'].to_numpy()[TE['idx']]
FINAL = {
    'random': np.zeros(len(te_u)),
    'popularity (orders, 365 d)': np.nan_to_num(pop, nan=-1.0),
    'LightGBM benchmark (retrained)': lgb_scores[LGB_BENCH]['test'],
    f'LightGBM tuned ({best_lgb})': LGB_SCORE['test'],
    f'neural, {len(SEEDS)}-seed average ({best_nn})': NN_SCORE['test'],
    f'ensemble ({best_how}, w={best_w})': ENS_SCORE['test'],
}
M = {n: slate_metrics(te_u, te_lab, s) for n, s in FINAL.items()}
final = pd.DataFrame({n: evaluate(te_u, te_lab, s) for n, s in FINAL.items()}).T
print(final[['NDCG@10', 'HR@10', 'Purchase-HR@10', 'median purchase rank']].round(4).to_string())

def paired(a, b, metric, n_boot=2000, seed=SEED):
    ma, mb = M[a], M[b]
    if metric == 'ndcg':
        d = ma['ndcg'] - mb['ndcg']
    else:
        keep = ma['has_buy']
        d = ma['buy_hit'][keep].astype(float) - mb['buy_hit'][keep].astype(float)
    rng = np.random.default_rng(seed)
    boot = np.concatenate([d[rng.integers(0, len(d), (200, len(d)))].mean(1) for _ in range(n_boot // 200)])
    return {'difference': d.mean(), 'ci_low': np.percentile(boot, 2.5), 'ci_high': np.percentile(boot, 97.5),
            'p (one-sided)': max((boot <= 0).mean(), 1 / n_boot)}

names = list(FINAL)
bench, lgb_t, nn_n, ens = names[2], names[3], names[4], names[5]
tests = {}
for a, b in [(ens, bench), (ens, lgb_t), (ens, nn_n), (nn_n, bench), (lgb_t, bench)]:
    for metric, label in [('ndcg', 'NDCG@10'), ('buy', 'Purchase-HR@10')]:
        tests[f'{a.split(" (")[0]} vs {b.split(" (")[0]} [{label}]'] = paired(a, b, metric)
print('\npaired bootstrap over test slates (positive difference = first model better):')
pd.DataFrame(tests).T.round(5)

                                              NDCG@10   HR@10  Purchase-HR@10  median purchase rank
random                                         0.0843  0.2283          0.1419                  44.0
popularity (orders, 365 d)                     0.2904  0.5639          0.5706                   8.0
LightGBM benchmark (retrained)                 0.4177  0.7228          0.6679                   5.0
LightGBM tuned (label_gain purchase=15)        0.4183  0.7233          0.6766                   5.0
neural, 3-seed average (listwise weight 2.0)   0.4216  0.7270          0.6750                   5.0
ensemble (z-score, w=0.6)                      0.4225  0.7271          0.6789                   5.0

paired bootstrap over test slates (positive difference = first model better):


,difference,ci_low,ci_high,p (one-sided)
ensemble vs LightGBM benchmark [NDCG@10],0.00483,0.00331,0.00626,0.0005
ensemble vs LightGBM benchmark [Purchase-HR@10],0.01102,0.00356,0.01815,0.0010
ensemble vs LightGBM tuned [NDCG@10],0.00414,0.00260,0.00557,0.0005
ensemble vs LightGBM tuned [Purchase-HR@10],0.00227,-0.00551,0.00940,0.2840
"ensemble vs neural, 3-seed average [NDCG@10]",0.00088,0.00011,0.00170,0.0100
"ensemble vs neural, 3-seed average [Purchase-HR@10]",0.00389,-0.00097,0.00875,0.0610
"neural, 3-seed average vs LightGBM benchmark [NDCG@10]",0.00395,0.00219,0.00559,0.0005
"neural, 3-seed average vs LightGBM benchmark [Purchase-HR@10]",0.00713,-0.00130,0.01555,0.0535
LightGBM tuned vs LightGBM benchmark [NDCG@10],0.00069,-0.00034,0.00175,0.0985
LightGBM tuned vs LightGBM benchmark [Purchase-HR@10],0.00875,0.00227,0.01523,0.0020


In [13]:
hist_wids = np.unique(pd.read_parquet(OUT_DIR / 'events.parquet', columns=['wid'])['wid'].to_numpy())
te_wid = feat['wid'].to_numpy()[TE['idx']]
j = np.searchsorted(hist_wids, te_wid)
cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != te_wid)
seg_user = pd.cut(pd.Series(np.where(te_lab > 0, cold, np.nan)).groupby(te_u).mean(),
                  [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
seg = pd.Series(te_u).map(seg_user).to_numpy()
seg_table = pd.DataFrame({n: {s: evaluate(te_u[seg == s], te_lab[seg == s], FINAL[n][seg == s])['NDCG@10']
                              for s in ['all warm', 'mixed', 'all cold']} for n in names[2:]}).T
print('test NDCG@10 by segment (slates whose positives are all warm / mixed / all cold items):')
seg_table.round(4)

test NDCG@10 by segment (slates whose positives are all warm / mixed / all cold items):


,all warm,mixed,all cold
LightGBM benchmark (retrained),0.5080,0.2738,0.1207
LightGBM tuned (label_gain purchase=15),0.5087,0.2736,0.1222
"neural, 3-seed average (listwise weight 2.0)",0.5110,0.2733,0.1331
"ensemble (z-score, w=0.6)",0.5129,0.2738,0.1292


## 9. Session conversion estimates and purchase ranks

The neural model's calibrated P(purchase) values give a session-level estimate of 1 − Π(1 − P(purchase)ᵢ), assuming independence between candidates. Comparing estimates with observed conversion by decile checks their calibration. Well-calibrated estimates could support promotion targeting for low-conversion sessions. Step 4c found that promotion-dependent product types converted 2.5 times less often.

For sessions containing a purchase, the ranking comparison reports Purchase-HR@10 and the median rank of the first purchased item. JD's original candidate order is excluded as a baseline because it leaks the labels.


In [14]:
P3 = NN_P['test'][:, 2]
st, sz = TE['start'], TE['size']
p_conv = 1 - np.exp(np.add.reduceat(np.log1p(-np.clip(P3, 0, 1 - 1e-7)), st))
obs_conv = np.maximum.reduceat((te_lab == 3).astype(int), st)
dec = pd.qcut(pd.Series(p_conv).rank(method='first'), 10, labels=[f'D{i}' for i in range(1, 11)])
calib = pd.DataFrame({'predicted': p_conv, 'observed': obs_conv, 'decile': dec}).groupby('decile', observed=True).agg(
    sessions=('observed', 'size'), predicted_conversion=('predicted', 'mean'), observed_conversion=('observed', 'mean'))
print(f'test sessions with an interaction: {len(st):,} | predicted conversion {p_conv.mean():.2%} vs observed {obs_conv.mean():.2%}')
print('by decile of predicted conversion (D10 = most likely to convert):')
print((calib * [1, 100, 100]).round(2).rename(columns=lambda c: c + (' %' if c != 'sessions' else '')).to_string())

biz = pd.DataFrame({n: {'purchased item in top 10': M[n]['buy_hit'][M[n]['has_buy']].mean(),
                        'median position of purchased item': np.nanmedian(M[n]['buy_rank']),
                        'purchased item at position 1': (M[n]['buy_rank'] == 1)[M[n]['has_buy']].mean()}
                    for n in names}).T
print(f"\nsessions that ended in a purchase: {int(M[names[0]]['has_buy'].sum()):,}")
biz.round(4)

test sessions with an interaction: 25,822 | predicted conversion 11.43% vs observed 11.95%
by decile of predicted conversion (D10 = most likely to convert):
        sessions  predicted_conversion %  observed_conversion %
decile                                                         
D1          2583                    3.75                   5.07
D2          2582                    5.92                   8.06
D3          2582                    7.23                  10.92
D4          2582                    8.32                  12.70
D5          2582                    9.38                  11.97
D6          2582                   10.62                  12.55
D7          2582                   12.24                  14.60
D8          2582                   14.42                  12.66
D9          2582                   17.75                  13.59
D10         2583                   24.67                  17.38

sessions that ended in a purchase: 3,086


,purchased item in top 10,median position of purchased item,purchased item at position 1
random,0.1419,44.0,0.0165
"popularity (orders, 365 d)",0.5706,8.0,0.1448
LightGBM benchmark (retrained),0.6679,5.0,0.2226
LightGBM tuned (label_gain purchase=15),0.6766,5.0,0.2216
"neural, 3-seed average (listwise weight 2.0)",0.6750,5.0,0.2252
"ensemble (z-score, w=0.6)",0.6789,5.0,0.2255


## 10. Save results

`step6_test_predictions_cps.parquet` contains the ensemble ranking in column `ensemble`, along with both models' scores for every test candidate. `step6_results_cps.csv` contains the result tables.


In [15]:
res = pd.concat([final.assign(stage='final test'),
                 pd.DataFrame(nn_grid).T.assign(stage='neural grid (val)'),
                 pd.DataFrame(lgb_grid).T.assign(stage='LightGBM grid (val)'),
                 pd.DataFrame(tests).T.assign(stage='paired bootstrap (test)')])
res.index.name = 'model / comparison'
pred = pd.DataFrame({'user': te_u, 'wid': te_wid, 'label': te_lab, 'lgbm': LGB_SCORE['test'], 'neural': NN_SCORE['test'],
                     'ensemble': ENS_SCORE['test'], 'p_click_plus': NN_P['test'][:, 0],
                     'p_cart_plus': NN_P['test'][:, 1], 'p_purchase': NN_P['test'][:, 2]})
for path, write in [(OUT_DIR / f'step6_results{OUT_SFX}.csv', lambda p: res.to_csv(p)),
                    (OUT_DIR / f'step6_test_predictions{OUT_SFX}.parquet', lambda p: pred.to_parquet(p, index=False))]:
    tmp = path.with_name('tmp_' + path.name)
    write(tmp); os.replace(tmp, path)
    print('saved', path)
try:
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass

saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step6_results_cps.csv
saved /content/drive/MyDrive/JDSearch/Raw Data/processed/step6_test_predictions_cps.parquet
Drive flushed and unmounted - safe to close the runtime


## 11. Results

The selected final model is the Part B ensemble, with test NDCG@10 of 0.4225. Both rankers use the Step 9 content-profile columns. Validation NDCG@10 increased from 0.4176 to 0.4219 for the neural ranker and from 0.4185 to 0.4226 for the ensemble, meeting the adoption criterion.

### Part A: LightGBM feature selection

| Variant | Val NDCG@10 | Test NDCG@10 | Test diff. vs baseline (95% CI) |
|---|---|---|---|
| baseline (55 columns) | 0.4153 | 0.4158 | |
| + all 13 | 0.4163 | 0.4178 | +0.0020 (0.0006 to 0.0033) |
| + all 13, − LSA (selected) | 0.4172 | 0.4183 | +0.0024 (0.0010 to 0.0038) |
| + top 5, − LSA | 0.4160 | 0.4165 | +0.0007 (−0.0007 to 0.0021) |

The selected feature set improves test NDCG@10 by 0.0024, close to Step 9's 300-tree estimate of 0.0027. Removing the three LSA profile columns also improves the result, consistent with the Step 4d audit. `cps_max`, `slate_pct_cps_max` and `slate_pct_cps_query` account for two thirds of the new columns' split gain.

On all-cold slates, LightGBM NDCG@10 increases from 0.1124 to 0.1223. The content profiles provide a similarity signal for candidates without exact-ID history.

### Part B: Final models on 25,822 test slates

| Model | NDCG@10 | HR@10 | Purchase-HR@10 | Purchased item at #1 | All-cold NDCG@10 |
|---|---|---|---|---|---|
| Popularity (orders, 365 d) | 0.2904 | 0.5639 | 0.5706 | 14.5% | — |
| LightGBM tuned (65 columns) | 0.4183 | 0.7233 | 0.6766 | 22.2% | 0.1222 |
| Neural, 3-seed average (37 features) | 0.4216 | 0.7270 | 0.6750 | 22.5% | 0.1331 |
| Ensemble (z-score, w = 0.6) | 0.4225 | 0.7271 | 0.6789 | 22.6% | 0.1292 |
| Step 6 ensemble, without the columns | 0.4201 | 0.7236 | 0.6756 | 22.8% | 0.1198 |

The neural ranker's test NDCG@10 increases by 0.0031 over Step 6's 0.4185, compared with LightGBM's gain of 0.0024. Its all-cold NDCG@10 increases from 0.124 to 0.133, the largest improvement on this segment in the project. The sparse TF-IDF profiles retain term-level matches that may be compressed in the 64-d LSA embeddings used by history attention.

Paired bootstrap comparisons favour the ensemble over tuned LightGBM by 0.0041 NDCG@10 (95% CI 0.0026 to 0.0056, p < 0.001) and over the neural ranker by 0.0009 (CI 0.0001 to 0.0017, p = 0.01). Validation selects a neural weight of 0.6, compared with 0.5 in Step 6. The blend adds less once the neural model improves.

The ensemble exceeds Step 6's 0.4201 by 0.0024. These runs cannot be compared with paired slate-level scores. A Step 6 rerun without the columns reached 0.4196, within 0.0005 of the original result. The observed gain is about five times this run-to-run difference; Part A provides the controlled paired evidence for LightGBM (+0.0024, with a confidence interval excluding zero).

Purchase-HR@10 is 0.6789, the highest recorded in the project. Its improvement of 0.0023 over tuned LightGBM is not significant (p = 0.28).

The neural configuration also changed: history length is 50 and listwise loss weight is 2.0, compared with 100 and 1.0 in Step 6. These settings were selected on validation, but they may account for part of the neural improvement, so the full difference cannot be attributed to the content-profile columns alone.

Predicted session conversion is 11.4%, compared with 11.95% observed. The top decile is overestimated at 24.7%, compared with 17.4% observed. The estimates rank sessions well but need recalibration before use as probabilities.

The final ranking is column `ensemble` in `step6_test_predictions_cps.parquet`. Run Steps 1 to 4b, 5, 9 and 11 to reproduce it. Step 6 records the model without the content-profile columns; Step 7's error analysis applies to that earlier model.
